# 轨迹过滤演示（classic 策略）

本 Notebook 生成一段合成轨迹，依次应用以下过滤器：
- FilterCstLatLon：重复经纬度屏蔽
- FilterCstPosition：alt/lat/lon 整体不变屏蔽
- FilterCstSpeed：vertical_rate/track/groundspeed 整体不变屏蔽
- MyFilterDerivative：基于一/二阶导阈值剔除突刺点（track 先 unwrap）
- FilterIsolated：≥20 秒的时间孤立点屏蔽

仅做“置 NaN/剔除”，不做插值；方便观察每一步的效果。

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datetime import timedelta

from filterclassic import (
    FilterCstLatLon, FilterCstPosition, FilterCstSpeed,
    MyFilterDerivative, FilterIsolated
)

pd.set_option('display.max_rows', 10)
plt.rcParams['figure.figsize'] = (10, 4)


ModuleNotFoundError: No module named 'filterclassic'

## 1. 生成合成轨迹
- 时长：120 秒（1Hz 采样）。
- 轨迹：经纬度缓慢变化，高度缓爬。
- 人为注入：重复经纬、速度常值段、单点突刺、跨 360° 的 `track` 跳变、以及变量级孤立点（便于被 FilterIsolated 捕捉）。

In [ ]:
n = 120
t0 = pd.Timestamp('2024-01-01 00:00:00', tz='UTC')
ts = pd.date_range(t0, periods=n, freq='s')

# 基础轨迹（平滑演示用）
lat0, lon0 = 30.0, 120.0
lat = lat0 + 0.001 * np.linspace(0, 1, n)
lon = lon0 + 0.001 * np.linspace(0, 1, n)
alt = 10000 + 20 * np.arange(n)  # ft
gs  = 250 + 0.5*np.sin(np.linspace(0, 6*np.pi, n))  # knots
trk = (90 + 0.1*np.arange(n)) % 360  # degrees

# 注入：重复经纬度段（20..25 与前一时刻相同）
lat[20:26] = lat[19]
lon[20:26] = lon[19]

# 注入：位置整体不变段（40..45）
alt[40:46] = alt[39]
lat[40:46] = lat[39]
lon[40:46] = lon[39]

# 注入：速度相关不变段（30..35）
gs[30:36] = gs[29]
trk[30:36] = trk[29]
# 垂直速度稍后按高度差计算，此段将体现为 0

# 注入：单点高度突刺（70）
alt[70] += 1000

# 注入：track 跨 360° 跳变（80..82）
trk[80] = 359
trk[81] = 1
trk[82] = 2

# 注入：latitude 变量级孤立点（仅 100 有值，附近全 NaN ≥ 25s）
lat_iso = lat.copy()
lat_iso[75:100] = np.nan
lat_iso[101:126] = np.nan  # 到末尾

# 计算垂直速度（ft/min），首个样本设为 NaN
vr = np.empty(n)
vr[:] = np.nan
vr[1:] = (alt[1:] - alt[:-1]) * 60

df = pd.DataFrame({
    'timestamp': ts,
    'flight_id': 1,
    'icao24': 0,
    'latitude': lat_iso,  # 用含孤立点的版本
    'longitude': lon,
    'altitude': alt,
    'vertical_rate': vr,
    'groundspeed': gs,
    'track': trk,
})
df.head(), df.tail(), df.shape


## 2. 依次应用过滤器（mask-only）

In [ ]:
def apply_filters(df_in: pd.DataFrame) -> pd.DataFrame:
    df1 = FilterCstLatLon().apply(df_in)
    df2 = FilterCstPosition().apply(df1)
    df3 = FilterCstSpeed().apply(df2)
    df4 = MyFilterDerivative().apply(df3)
    df5 = FilterIsolated().apply(df4)
    return df5

df_f = apply_filters(df)
df_f.head(), df_f.tail()


## 3. 前后对比（NaN 数量与可视化）

In [ ]:
def nan_summary(df0: pd.DataFrame, df1: pd.DataFrame):
    cols = ['latitude','longitude','altitude','vertical_rate','groundspeed','track']
    s0 = df0[cols].isna().sum().rename('before')
    s1 = df1[cols].isna().sum().rename('after')
    return pd.concat([s0, s1], axis=1)

nan_summary(df, df_f)


In [ ]:
fig, axes = plt.subplots(3, 1, sharex=True, figsize=(10, 6))
axes[0].plot(df['timestamp'], df['altitude'], label='altitude (raw)', alpha=0.6)
axes[0].plot(df_f['timestamp'], df_f['altitude'], label='altitude (filtered)')
axes[0].set_ylabel('ft')
axes[0].legend()

axes[1].plot(df['timestamp'], df['groundspeed'], label='gs (raw)', alpha=0.6)
axes[1].plot(df_f['timestamp'], df_f['groundspeed'], label='gs (filtered)')
axes[1].set_ylabel('kt')
axes[1].legend()

axes[2].plot(df['timestamp'], df['track'], label='track (raw)', alpha=0.6)
axes[2].plot(df_f['timestamp'], df_f['track'], label='track (filtered)')
axes[2].set_ylabel('deg')
axes[2].legend()
plt.show()


## 4. 备注与脚本调用
- 此处演示为“直接调用各 Filter 的 `apply`”，方便离线复现。
- 在实际流程中，使用 `traffic.core.Traffic(...).filter(filter=filter_chain, strategy=nointerpolate)` 的方式，见 `filter_trajs.py`。
- 命令行入口：`python3 filter_trajs.py -t_in <raw> -t_out <filtered> -strategy classic`。